In [3]:
import duckdb

raw_glob = "/storage/vacoda/data/raw/api/single/**/*.csv"

con = duckdb.connect()

columns = con.execute(
    """
    DESCRIBE
    SELECT *
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        sample_size = 100000
    )
    """,
    [raw_glob],
).df()

columns

,column_name,column_type,null,key,default,extra
0,TypeCode,VARCHAR,YES,None,None,None
1,FreqCode,VARCHAR,YES,None,None,None
2,RefPeriodId,BIGINT,YES,None,None,None
3,RefYear,BIGINT,YES,None,None,None
4,RefMonth,BIGINT,YES,None,None,None
5,Period,BIGINT,YES,None,None,None
6,ReporterCode,BIGINT,YES,None,None,None
7,ReporterISO,VARCHAR,YES,None,None,None
8,ReporterDesc,VARCHAR,YES,None,None,None
9,FlowCode,VARCHAR,YES,None,None,None


In [2]:
raw_codes = con.execute(
    """
    SELECT
        Period,
        ClassificationCode,
        CmdCode,
        ReporterISO,
        PartnerISO,
        FlowCode,
        NetWgt,
        PrimaryValue,
        IsReported,
        filename
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        filename = true,
        sample_size = 100000
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0')
          IN ('440723', '440729')
    ORDER BY
        CmdCode,
        Period,
        ReporterISO,
        PartnerISO,
        FlowCode
    """,
    [raw_glob],
).df()

raw_codes

,Period,ClassificationCode,CmdCode,ReporterISO,PartnerISO,FlowCode,NetWgt,PrimaryValue,IsReported,filename
0,1988,H0,440723,AUS,BOL,M,16800.00,11644.000,False,/storage/vacoda/data/raw/api/single/comtrade_4...
1,1988,H0,440723,AUS,BRA,M,539000.00,478076.000,False,/storage/vacoda/data/raw/api/single/comtrade_4...
2,1988,H0,440723,AUS,CHL,M,16101.00,9321.000,False,/storage/vacoda/data/raw/api/single/comtrade_4...
3,1988,H0,440723,AUS,ECU,M,180601.00,105423.000,False,/storage/vacoda/data/raw/api/single/comtrade_4...
4,1988,H0,440723,AUS,FJI,M,174300.00,91877.000,False,/storage/vacoda/data/raw/api/single/comtrade_4...
...,...,...,...,...,...,...,...,...,...,...
274856,2025,H6,440729,ZAF,ZMB,M,0.00,41300.819,False,/storage/vacoda/data/raw/api/single/comtrade_4...
274857,2025,H6,440729,ZAF,ZMB,X,989.69,1489.305,False,/storage/vacoda/data/raw/api/single/comtrade_4...
274858,2025,H6,440729,ZAF,ZMB,X,989.69,1489.305,False,/storage/vacoda/data/raw/api/single/comtrade_4...
274859,2025,H6,440729,ZAF,ZMB,X,989.69,1489.305,False,/storage/vacoda/data/raw/api/single/comtrade_4...


In [3]:
code_identity = con.execute(
    """
    SELECT
        LPAD(CAST(CmdCode AS VARCHAR), 6, '0') AS raw_cmd,
        ClassificationCode                    AS raw_hs_version,
        MIN(Period)                            AS first_year,
        MAX(Period)                            AS last_year,
        COUNT(*)                               AS n_rows,
        COUNT(DISTINCT ReporterISO)             AS n_reporters,
        COUNT(DISTINCT PartnerISO)              AS n_partners,
        SUM(PrimaryValue)                       AS total_primary_value,
        SUM(NetWgt)                             AS total_net_weight
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        sample_size = 100000
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0')
          IN ('440723', '440729')
    GROUP BY
        raw_cmd,
        raw_hs_version
    ORDER BY
        raw_cmd,
        first_year
    """,
    [raw_glob],
).df()

code_identity

,raw_cmd,raw_hs_version,first_year,last_year,n_rows,n_reporters,n_partners,total_primary_value,total_net_weight
0,440723,H0,1988,2007,4587,114,183,3.490244e+09,6.962464e+09
1,440723,H6,2022,2025,12595,102,157,5.161818e+09,3.647067e+09
2,440729,H1,1996,2014,22068,160,225,3.493254e+10,2.693807e+11
3,440729,H2,2002,2018,37699,162,230,6.634464e+10,7.575442e+10
4,440729,H3,2007,2025,39419,165,231,5.438154e+10,6.169820e+10
5,440729,H4,2012,2024,41961,166,226,4.465574e+10,5.000950e+10
6,440729,H5,2017,2025,67115,165,224,1.536156e+11,2.484130e+11
7,440729,H6,2022,2025,49417,139,216,1.054855e+11,2.280068e+11


In [4]:
files_with_codes = con.execute(
    """
    SELECT
        LPAD(CAST(CmdCode AS VARCHAR), 6, '0') AS raw_cmd,
        ClassificationCode,
        Period,
        filename,
        COUNT(*) AS n_rows
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        filename = true,
        sample_size = 100000
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0')
          IN ('440723', '440729')
    GROUP BY
        raw_cmd,
        ClassificationCode,
        Period,
        filename
    ORDER BY
        raw_cmd,
        Period,
        filename
    """,
    [raw_glob],
).df()

files_with_codes

,raw_cmd,ClassificationCode,Period,filename,n_rows
0,440723,H0,1988,/storage/vacoda/data/raw/api/single/comtrade_4...,147
1,440723,H0,1989,/storage/vacoda/data/raw/api/single/comtrade_4...,301
2,440723,H0,1990,/storage/vacoda/data/raw/api/single/comtrade_4...,317
3,440723,H0,1991,/storage/vacoda/data/raw/api/single/comtrade_4...,401
4,440723,H0,1992,/storage/vacoda/data/raw/api/single/comtrade_4...,493
...,...,...,...,...,...
100,440729,H3,2024,/storage/vacoda/data/raw/api/single/comtrade_4...,46
101,440729,H5,2024,/storage/vacoda/data/raw/api/single/comtrade_4...,130
102,440729,H3,2025,/storage/vacoda/data/raw/api/single/comtrade_4...,4
103,440729,H6,2025,/storage/vacoda/data/raw/api/single/comtrade_4...,8576


In [5]:
files_with_codes = con.execute(
    """
    SELECT
        LPAD(CAST(CmdCode AS VARCHAR), 6, '0') AS raw_cmd,
        ClassificationCode,
        Period,
        regexp_extract(filename, '[^/]+$') AS file_name,
        COUNT(*) AS n_rows
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        filename = true,
        sample_size = 100000
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0')
          IN ('440723', '440729')
    GROUP BY
        raw_cmd,
        ClassificationCode,
        Period,
        file_name
    ORDER BY
        raw_cmd,
        Period,
        ClassificationCode,
        file_name
    """,
    [raw_glob],
).df()

In [6]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

files_with_codes.head(30)

,raw_cmd,ClassificationCode,Period,file_name,n_rows
0,440723,H0,1988,comtrade_440723_1988.csv,147
1,440723,H0,1989,comtrade_440723_1989.csv,301
2,440723,H0,1990,comtrade_440723_1990.csv,317
3,440723,H0,1991,comtrade_440723_1991.csv,401
4,440723,H0,1992,comtrade_440723_1992.csv,493
5,440723,H0,1993,comtrade_440723_1993.csv,617
6,440723,H0,1994,comtrade_440723_1994.csv,800
7,440723,H0,1995,comtrade_440723_1995.csv,887
8,440723,H0,1996,comtrade_440723_1996.csv,262
9,440723,H0,1997,comtrade_440723_1997.csv,82


In [4]:
h6_440729_files = con.execute(
    """
    SELECT
        filename,
        Period,
        COUNT(*) AS n_rows
    FROM read_csv_auto(
        ?,
        union_by_name = true,
        filename = true,
        sample_size = -1
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0') = '440729'
      AND ClassificationCode = 'H6'
    GROUP BY filename, Period
    ORDER BY Period, filename
    """,
    [raw_glob],
).df()

print(h6_440729_files.to_string(index=False))

                                                    filename  Period  n_rows
/storage/vacoda/data/raw/api/single/comtrade_440729_2022.csv    2022   13869
/storage/vacoda/data/raw/api/single/comtrade_440729_2023.csv    2023   13519
/storage/vacoda/data/raw/api/single/comtrade_440729_2024.csv    2024   13453
/storage/vacoda/data/raw/api/single/comtrade_440729_2025.csv    2025    8576


In [5]:
file_path = (
    "/storage/vacoda/data/raw/api/single/"
    "comtrade_440729_2025.csv"
)

evidence = con.execute(
    """
    SELECT
        Period,
        ClassificationCode,
        CmdCode,
        ReporterISO,
        PartnerISO,
        FlowCode,
        PrimaryValue,
        NetWgt
    FROM read_csv_auto(
        ?,
        sample_size = -1
    )
    WHERE LPAD(CAST(CmdCode AS VARCHAR), 6, '0') = '440729'
      AND ClassificationCode = 'H6'
    LIMIT 20
    """,
    [file_path],
).df()

print(evidence.to_string(index=False))

 Period ClassificationCode  CmdCode ReporterISO PartnerISO FlowCode  PrimaryValue    NetWgt
   2025                 H6   440729         BLZ        FRA        X      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA        X      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA        X      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA        X      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA       DX      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA       DX      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA       DX      62573.50 10204.300
   2025                 H6   440729         BLZ        FRA       DX      62573.50 10204.300
   2025                 H6   440729         BLZ        USA        X      96997.85 11366.721
   2025                 H6   440729         BLZ        USA        X      96997.8

## Product-group

In [1]:
import pandas as pd

path = "/storage/vacoda/data/interim/cif_fob/target_h5/target_h5_cif_fob_training_sample.parquet"

df = pd.read_parquet(path)

print(df.columns.tolist())

['year', 'exporter_iso', 'importer_iso', 'target_cmd', 'export_value_fob', 'import_value_cif', 'export_weight_kg_pre_allocation', 'import_weight_kg_pre_allocation', 'export_weight_kg', 'import_weight_kg', 'export_weight_source', 'import_weight_source', 'export_weight_estimated', 'import_weight_estimated', 'export_weight_is_proxy', 'import_weight_is_proxy', 'import_direct_cif_value', 'import_direct_fob_value', 'import_direct_cif_fob_factor', 'import_direct_covered_primary_value', 'import_direct_value_coverage', 'import_direct_pair_component_count', 'export_component_row_count', 'import_component_row_count', 'export_partner2_count', 'import_partner2_count', 'export_mot_count', 'import_mot_count', 'export_customs_count', 'import_customs_count', 'export_mos_count', 'import_mos_count', 'export_source_hs_count', 'import_source_hs_count', 'export_source_cmd_count', 'import_source_cmd_count', 'export_source_cmd_original_count', 'import_source_cmd_original_count', 'export_estimated_weight_compo

In [2]:
"product_group" in df.columns

False

In [3]:
[c for c in df.columns if "cmd" in c.lower() or "hs" in c.lower()]

['target_cmd',
 'export_source_hs_count',
 'import_source_hs_count',
 'export_source_cmd_count',
 'import_source_cmd_count',
 'export_source_cmd_original_count',
 'import_source_cmd_original_count']

In [4]:
df["hs4_product_group"] = (
    df["target_cmd"]
    .astype(str)
    .str.zfill(6)
    .str[:4]
)

In [5]:
df[["target_cmd", "hs4_product_group"]].drop_duplicates().head(5)

,target_cmd,hs4_product_group
0,441239,4412
28,441875,4418
64,441879,4418
65,441873,4418
133,480258,4802


In [6]:
df["hs4_product_group"].nunique()

86

In [7]:
df["hs4_product_group"].value_counts().head(10)

hs4_product_group
9403    297304
4819    288035
4820    216147
4802    212580
4911    208659
4823    200914
9401    185533
4811    184448
4901    178316
4418    174339
Name: count, dtype: int64

In [8]:
df["hs4_product_group"].value_counts().tail(20)

hs4_product_group
4904    11901
5504    11685
5510    11349
3921    11259
5502     8395
4406     7226
4405     7058
5508     6873
4704     6579
5511     6297
9620     5240
4701     4434
4702     4236
4705     4040
5405     3914
5507     3553
5505     2262
9619      882
3006       19
5406        8
Name: count, dtype: int64